# Sentence Classification Generator

### Authors - Shivendra Raghav (1001123853), Amirhosein Morteza (1002187551)
### CSE 5388-002

#### This files reads converts all the text files for processed synthetic dataset into a single CSV file with classification columns and metadata.

### Import the required libraries

In [1]:
import os
import json
import spacy
import random
import pandas as pd

# spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

### Read all the files into a single dataframe

In [2]:
# File path for the processed dataset
processed_sensitive_with_placeholders_path = "../synthetic_dataset/processed_sensitive_with_placeholders"

df = pd.DataFrame(columns=["filename", "text"])

# Read the processed dataset
for filename in os.listdir(processed_sensitive_with_placeholders_path):
    with open(os.path.join(processed_sensitive_with_placeholders_path, filename), "r") as f:
        text = f.read()
        df = df._append({"filename": filename, "text": text}, ignore_index=True)

df.head()

,filename,text
0,email_enron_jones-t_sent_2182_prompt_1_not_pro...,Message-ID: <10746365.1075847203014.JavaMail.e...
1,technology_prompt_storage_9_1_1.txt,Oracle Cloud Storage Documentation\n\nIntroduc...
2,technology_prompt_onboarding_17_1_1.txt,```\nData Analyst Onboarding Documentation\n\n...
3,email_enron_taylor-m_sent_799_prompt_1_not_pro...,<email>Message-ID: <20950274.1075860067027.Jav...
4,business_prompt_14_1_1.txt,ABC Corp Employee Training Manual\n\nIntroduct...


### Convert the text content into sentences with classification

In [3]:
sensitive_placeholders = ["<ACCESS_KEY>", "<PRIMARY_ACCESS_KEY>", "<ACCESS_TOKEN>", "<API_KEY>", "<API_SECRET>", "<API_TOKEN>", "<BANK_ACCOUNT_NUMBER>", "<BEARER_TOKEN>", "<DATABASE_CONN>", "<INTERNATIONAL_BANK_ACCOUNT_NUMBER>", "<LICENSE_PLATE>", 
"<OAUTH_ID>", "<OAUTH_SECRET>", "<PASSWORD>", "<SSN>", "<USERNAME>", "<CREDIT_CARD_NUMBER>",  "<AZURE_SQL_DATABASE_CONN>", "<AWS_DYNAMODB_DATABASE_CONN>", "<GCP_SQL_DATABASE_CONN>", 
"<GOOGLE_FIRESTORE_DATABASE_CONN>", "<MONGODB_DATABASE_CONN>", "<GCP_POSTGRES_DATABASE_CONN>", "<POSTGRESQL_DATABASE_CONN>", "<MYSQL_DATABASE_CONN>", "<ORACLE_DATABASE_CONN>", "<ELASTISEARCH_DATABASE_CONN>", "<NEO4J_DATABASE_CONN>", "<CASSANDRA_DATABASE_CONN>", "<REDIS_DATABASE_CONN>", "<MSFT_SQL_DATABASE_CONN>", "<IBM_DB2_DATABASE_CONN>", "<COUCHBASE_DATABASE_CONN>", "<SECRET_KEY>"]

In [4]:
# Sentence Classifier Dataframe
sentence_df = pd.DataFrame(columns=["filename", "sentence", "sensitive"])

# Iterate over the processed dataset
for index, row in df.iterrows():
    filename = row["filename"]
    text = row["text"]
    content = nlp(text)
    sentences = [sent.text.strip() for sent in content.sents]

    for sentence in sentences:
        # Check if sentence is empty
        if sentence == "":
            continue
        else:
            # Check if sentence has a sensitive placeholder
            sensitive = 0
            for placeholder in sensitive_placeholders:
                if placeholder in sentence:
                    sensitive = 1
                    break
            sentence_df = sentence_df._append({"filename": filename, "sentence": sentence, "sensitive": sensitive}, ignore_index=True)
        
sentence_df.head()

,filename,sentence,sensitive
0,email_enron_jones-t_sent_2182_prompt_1_not_pro...,Message-ID: <10746365.1075847203014.JavaMail.e...,0
1,email_enron_jones-t_sent_2182_prompt_1_not_pro...,"To: david.minns@enron.com, jane.mcbride@enron....",0
2,email_enron_jones-t_sent_2182_prompt_1_not_pro...,I hope this email finds you well.,0
3,email_enron_jones-t_sent_2182_prompt_1_not_pro...,Please find the access details for the ISDA Ma...,1
4,email_enron_jones-t_sent_2182_prompt_1_not_pro...,You should now have edit access to the referen...,0


In [5]:
# Drop the original datframe as we no longer need it
df.drop(df.index, inplace=True)

# Drop empty sentences from the sentence dataframe
sentence_df = sentence_df[sentence_df["sentence"] != ""]

### Add synthetic secrets and NER metadata

In [6]:
# Load the secrets datasets
secrets = json.load(open("../sensitive_data_values/secrets.json"))

# Load synthetic user profiles
user_profiles_df = pd.read_csv("../sensitive_data_values/profiles.csv")

sensitive_values = {
    "<PASSWORD>": secrets["Passwords"],
    "<USERNAME>": user_profiles_df["username"].tolist(),
    "<SSN>": user_profiles_df["ssn"].tolist(),
    "<BANK_ACCOUNT_NUMBER>": user_profiles_df["bank_account_number"].tolist(),
    "<INTERNATIONAL_BANK_ACCOUNT_NUMBER>": user_profiles_df["international_bank_account_number"].tolist(),
    "<LICENSE_PLATE>": user_profiles_df["license_plate"].tolist(),
    "<CREDIT_CARD_NUMBER>": user_profiles_df["credit_card_number"].tolist(),
    "<ACCESS_TOKEN>": secrets["Access_Token"],
    "<BEARER_TOKEN>": secrets["Access_Token"],
    "<ACCESS_KEY>": secrets["Access_Token"],
    "<API_KEY>": secrets["API_Key"],
    "<API_SECRET>": secrets["API_Secret"],
    "<API_TOKEN>": secrets["API_Token"],
    "<OAUTH_ID>": secrets["OAuth_Id"],
    "<OAUTH_SECRET>": secrets["OAuth_Secret"],
    "<SECRET_KEY>": secrets["API_Secret"],
    "<MONGODB_DATABASE_CONN>": secrets["MongoDB_Conn"],
    "<AZURE_SQL_DATABASE_CONN>": secrets["Azure_SQL_Conn"],
    "<AWS_DYNAMODB_DATABASE_CONN>": secrets["AWS_DynamoDB_Conn"],
    "<GCP_SQL_DATABASE_CONN>": secrets["GCP_SQL_Conn"],
    "<GOOGLE_FIRESTORE_DATABASE_CONN>": secrets["Google_Firestore_Conn"],
    "<GCP_POSTGRES_DATABASE_CONN>": secrets["GCP_Postgres_Conn"],
    "<POSTGRESQL_DATABASE_CONN>": secrets["PostgreSQL_Conn"],
    "<MYSQL_DATABASE_CONN>": secrets["MySQL_Conn"],
    "<ORACLE_DATABASE_CONN>": secrets["Oracle_Conn"],
    "<ELASTISEARCH_DATABASE_CONN>": secrets["Elasticsearch_Conn"],
    "<NEO4J_DATABASE_CONN>": secrets["Neo4j_Conn"],
    "<CASSANDRA_DATABASE_CONN>": secrets["Cassandra_Conn"],
    "<REDIS_DATABASE_CONN>": secrets["Redis_Conn"],
    "<MSFT_SQL_DATABASE_CONN>": secrets["Microsoft_SQL_Conn"],
    "<IBM_DB2_DATABASE_CONN>": secrets["IBM_Db2_Conn"],
    "<COUCHBASE_DATABASE_CONN>": secrets["Couchbase_Conn"]
    }

database_conn = sensitive_values["<MONGODB_DATABASE_CONN>"].copy()
database_conn.extend(sensitive_values["<AZURE_SQL_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<AWS_DYNAMODB_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<GCP_SQL_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<GOOGLE_FIRESTORE_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<GCP_POSTGRES_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<POSTGRESQL_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<MYSQL_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<ORACLE_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<ELASTISEARCH_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<NEO4J_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<CASSANDRA_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<REDIS_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<MSFT_SQL_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<IBM_DB2_DATABASE_CONN>"])
database_conn.extend(sensitive_values["<COUCHBASE_DATABASE_CONN>"])

sensitive_values["<DATABASE_CONN>"] = database_conn

### Replace Sensitive Placeholders with Synthetic Secrets

In [7]:
# Sentence Classifier Dataframe with Synthetic Secrets
sentence_secrets_df = pd.DataFrame(columns=["filename", "sentence_with_placeholder", "sentence", "sensitive", "ner_metadata"])

# Iterate over the sentence dataframe
for index, row in sentence_df.iterrows():
    filename = row["filename"]
    sentence_with_placeholder = row["sentence"]
    sensitive = row["sensitive"]
    ner_metadata = [] 

    # Check if sentence is sensitive
    if sensitive == 1:
        sentence = sentence_with_placeholder
        for placeholder in sensitive_placeholders:
            if placeholder in sentence:
                value = str(random.choice(sensitive_values[placeholder]))
                sentence = sentence.replace(placeholder, value)
                ner_metadata.append((value, {"entities": [(sentence.index(value), sentence.index(value) + len(value), placeholder[1:-1])]}))
    else:
        sentence = sentence_with_placeholder

    sentence_secrets_df = sentence_secrets_df._append({"filename": filename, "sentence_with_placeholder": sentence_with_placeholder, "sentence": sentence, "sensitive": sensitive, "ner_metadata": ner_metadata}, ignore_index=True)

sentence_secrets_df.head()

,filename,sentence_with_placeholder,sentence,sensitive,ner_metadata
0,email_enron_jones-t_sent_2182_prompt_1_not_pro...,Message-ID: <10746365.1075847203014.JavaMail.e...,Message-ID: <10746365.1075847203014.JavaMail.e...,0,[]
1,email_enron_jones-t_sent_2182_prompt_1_not_pro...,"To: david.minns@enron.com, jane.mcbride@enron....","To: david.minns@enron.com, jane.mcbride@enron....",0,[]
2,email_enron_jones-t_sent_2182_prompt_1_not_pro...,I hope this email finds you well.,I hope this email finds you well.,0,[]
3,email_enron_jones-t_sent_2182_prompt_1_not_pro...,Please find the access details for the ISDA Ma...,Please find the access details for the ISDA Ma...,1,"[(H-oex9tb4EOetNtymPQ0W7UsaSVqtHivmp1XzpgP, {'..."
4,email_enron_jones-t_sent_2182_prompt_1_not_pro...,You should now have edit access to the referen...,You should now have edit access to the referen...,0,[]


In [8]:
# Export to CSV
sentence_secrets_df.to_csv("../synthetic_dataset/sentence_classified_data.csv", index=False)